<a href="https://colab.research.google.com/github/burakderee/siir-olusturucu/blob/main/gpt2_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_adi = "ytu-ce-cosmos/turkish-gpt2"  # YTÜ Türkçe GPT-2

tokenizer = AutoTokenizer.from_pretrained(model_adi)
model = AutoModelForCausalLM.from_pretrained(model_adi)
model = model.cuda()

print("✅ GPT-2 Türkçe yüklendi.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
from torch.utils.data import Dataset, DataLoader
import torch

# Veriyi yükle
ORIJINAL_YOL = "/content/drive/MyDrive/orhan_veli_dogru_format.jsonl"
SENTETIK_YOL = "/content/drive/MyDrive/orhan_veli_sentetik.jsonl"

veriler = []
for yol in [ORIJINAL_YOL, SENTETIK_YOL]:
    with open(yol, "r", encoding="utf-8") as f:
        for satir in f:
            satir = satir.strip()
            if satir:
                ornek = json.loads(satir)
                metin = f"### Talimat:\nSen Orhan Veli Kanık'sın. \"{ornek['input']}\" kelimesini kullanarak şiir yaz.\n\n### Şiir:\n{ornek['output']}"
                veriler.append(metin)

print(f"✅ Toplam {len(veriler)} örnek yüklendi.")

class SiirDataset(Dataset):
    def __init__(self, metinler, tokenizer, max_len=512):
        self.ornekler = []
        for metin in metinler:
            enc = tokenizer(
                metin,
                truncation=True,
                max_length=max_len,
                padding="max_length",
                return_tensors="pt"
            )
            self.ornekler.append({
                "input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels": enc["input_ids"].squeeze()
            })

    def __len__(self):
        return len(self.ornekler)

    def __getitem__(self, idx):
        return self.ornekler[idx]

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = SiirDataset(veriler, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"✅ Dataset hazır: {len(dataset)} örnek")

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=10,
    num_training_steps=len(dataloader) * 5
)

model.train()
kayiplar = []

for epoch in range(5):
    toplam_kayip = 0
    for batch in dataloader:
        input_ids      = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()
        labels         = batch["labels"].cuda()

        optimizer.zero_grad()
        cikti = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = cikti.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        toplam_kayip += loss.item()

    ort_kayip = toplam_kayip / len(dataloader)
    kayiplar.append(ort_kayip)
    print(f"Epoch {epoch+1}/5 — Loss: {ort_kayip:.4f}")

print(f"\n✅ Eğitim bitti. Final Loss: {kayiplar[-1]:.4f}")

In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, math

orijinal = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_dogru_format.jsonl")["train"]
sentetik  = load_dataset("json", data_files="/content/drive/MyDrive/orhan_veli_sentetik_v3.jsonl")["train"]
dataset_raw = concatenate_datasets([orijinal, sentetik])

split   = dataset_raw.train_test_split(test_size=0.2, seed=42)
val_set = split["test"]
print(f"Validation set: {len(val_set)} örnek")

# GPT-2 modeli yükle
gpt2_tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
)
gpt2_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
).cuda()

SABLON = "### Talimat:\nSen Orhan Veli Kanık'sın. \"{input}\" kelimesini kullanarak şiir yaz.\n\n### Şiir:\n{output}"

gpt2_model.eval()
toplam_loss = 0
sayi = 0

for ornek in val_set:
    metin = SABLON.format(input=ornek["input"], output=ornek["output"])
    enc = gpt2_tokenizer(metin, return_tensors="pt",
                         truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        cikti = gpt2_model(**enc, labels=enc["input_ids"])
        toplam_loss += cikti.loss.item()
        sayi += 1

val_loss = toplam_loss / sayi
val_perplexity = math.exp(val_loss)
train_loss = 0.2694

print(f"\n{'='*40}")
print(f"  GPT-2 VALIDATION SONUÇLARI")
print(f"{'='*40}")
print(f"  Validation Loss       : {val_loss:.4f}")
print(f"  Validation Perplexity : {val_perplexity:.2f}")
print(f"  Train Loss            : {train_loss}")
print(f"  Fark                  : {val_loss - train_loss:.4f}")
print(f"{'='*40}")

if val_loss - train_loss > 0.3:
    print("⚠️  Overfitting var")
else:
    print("✅ Overfitting yok")

In [ ]:
import os
kayit_yolu = "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
os.makedirs(kayit_yolu, exist_ok=True)

model.save_pretrained(kayit_yolu)
tokenizer.save_pretrained(kayit_yolu)

# Metrikleri kaydet
metrikler = {
    "model": "turkish-gpt2",
    "learning_rate": 5e-5,
    "epoch": 5,
    "train_loss": kayiplar[-1],
    "tum_kayiplar": kayiplar
}
with open(f"{kayit_yolu}/metrikler.json", "w") as f:
    json.dump(metrikler, f, indent=2)

print("✅ GPT-2 modeli kaydedildi.")

In [ ]:
import json
import torch
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    get_cosine_schedule_with_warmup
)
from torch.utils.data import DataLoader

# Daha önce tanımladığın SiirDataset sınıfı hâlâ bellekte olmalı

DENEYLER = [
    {"isim": "A", "lr": 1e-4, "epoch": 3, "batch": 4},
    {"isim": "B", "lr": 2e-4, "epoch": 5, "batch": 8},
    {"isim": "C", "lr": 5e-4, "epoch": 8, "batch": 4},
]

gpt2_sonuclar = []

for deney in DENEYLER:
    print(f"\n{'='*50}")
    print(f"DENEY {deney['isim']} — lr={deney['lr']} epoch={deney['epoch']} batch={deney['batch']}")
    print('='*50)

    # Modeli sıfırdan yükle — her deney temiz başlasın
    model_temp = AutoModelForCausalLM.from_pretrained("ytu-ce-cosmos/turkish-gpt2").cuda()

    dataloader_temp = DataLoader(dataset, batch_size=deney["batch"], shuffle=True)

    optimizer = AdamW(model_temp.parameters(), lr=deney["lr"])
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=5,
        num_training_steps=len(dataloader_temp) * deney["epoch"]
    )

    model_temp.train()
    kayiplar = []

    for epoch in range(deney["epoch"]):
        toplam = 0
        for batch in dataloader_temp:
            input_ids      = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()
            labels         = batch["labels"].cuda()

            optimizer.zero_grad()
            cikti = model_temp(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            cikti.loss.backward()
            optimizer.step()
            scheduler.step()
            toplam += cikti.loss.item()

        ort = toplam / len(dataloader_temp)
        kayiplar.append(ort)
        print(f"  Epoch {epoch+1}/{deney['epoch']} — Loss: {ort:.4f}")

    sonuc = {
        "model": "gpt2-turkish",
        "deney": deney["isim"],
        "lr": deney["lr"],
        "epoch": deney["epoch"],
        "batch": deney["batch"],
        "final_loss": kayiplar[-1],
        "tum_kayiplar": kayiplar
    }
    gpt2_sonuclar.append(sonuc)
    print(f"\n✅ Deney {deney['isim']} bitti. Final Loss: {kayiplar[-1]:.4f}")

    # En iyi modeli kaydet
    en_iyi = min(gpt2_sonuclar, key=lambda x: x["final_loss"])
    if sonuc["deney"] == en_iyi["deney"]:
        model_temp.save_pretrained("/content/drive/MyDrive/gpt2_orhan_veli_modeli")
        print(f"  💾 En iyi model kaydedildi.")

    del model_temp
    torch.cuda.empty_cache()

# Sonuçları kaydet
with open("/content/drive/MyDrive/gpt2_hiperparametre_sonuclari.json", "w") as f:
    json.dump(gpt2_sonuclar, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print("GPT-2 HİPERPARAMETRE OPTİMİZASYONU TAMAMLANDI")
print('='*50)
for s in gpt2_sonuclar:
    print(f"  Deney {s['deney']}: lr={s['lr']} epoch={s['epoch']} → Loss={s['final_loss']:.4f}")

en_iyi = min(gpt2_sonuclar, key=lambda x: x["final_loss"])
print(f"\n🏆 En iyi: Deney {en_iyi['deney']} — Loss={en_iyi['final_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json

# ── 1. OVERFİTTİNG ANALİZİ ──────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

renkler = {"A": "blue", "B": "green", "C": "red"}
basliklar = {
    "A": "Deney A\nlr=1e-4, epoch=3",
    "B": "Deney B\nlr=2e-4, epoch=5",
    "C": "Deney C\nlr=5e-4, epoch=8"
}

for i, sonuc in enumerate(gpt2_sonuclar):
    kayiplar = sonuc["tum_kayiplar"]
    ax = axes[i]
    ax.plot(range(1, len(kayiplar)+1), kayiplar,
            color=renkler[sonuc["deney"]],
            marker='o', linewidth=2, markersize=6)

    # Overfitting tespiti — son 3 epoch'ta artış var mı?
    if len(kayiplar) >= 3:
        son_uc = kayiplar[-3:]
        if son_uc[-1] > son_uc[0]:
            ax.axvspan(len(kayiplar)-2, len(kayiplar),
                      alpha=0.2, color='red', label='Overfitting bölgesi')
            ax.legend(fontsize=8)

    ax.set_title(basliklar[sonuc["deney"]], fontsize=10, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training Loss")
    ax.grid(True, alpha=0.3)

    # En iyi noktayı işaretle
    en_iyi_idx = np.argmin(kayiplar)
    ax.scatter(en_iyi_idx+1, kayiplar[en_iyi_idx],
               color='gold', s=150, zorder=5,
               marker='*', label=f'En iyi: {kayiplar[en_iyi_idx]:.4f}')
    ax.legend(fontsize=8)

plt.suptitle("GPT-2 Türkçe — Overfitting Analizi", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/gpt2_overfitting_analizi.png", dpi=150)
plt.show()
print("✅ Overfitting analizi kaydedildi.")

# ── 2. BAYESIAN OPTİMİZASYON ────────────────────────────────

!pip install bayesian-optimization -q

from bayes_opt import BayesianOptimization
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, get_cosine_schedule_with_warmup
import torch

bayesian_sonuclar = []

def gpt2_egit(learning_rate, epoch, batch_size):
    """Bayesian optimizer'ın çağıracağı hedef fonksiyon."""
    epoch      = int(round(epoch))
    batch_size = int(round(batch_size))
    batch_size = max(2, min(batch_size, 8))  # 2-8 arası sınırla

    model_temp = AutoModelForCausalLM.from_pretrained(
        "ytu-ce-cosmos/turkish-gpt2"
    ).cuda()

    dl = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(model_temp.parameters(), lr=learning_rate)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=3,
        num_training_steps=len(dl) * epoch
    )

    model_temp.train()
    son_loss = 0

    for ep in range(epoch):
        toplam = 0
        for batch in dl:
            input_ids      = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()
            labels         = batch["labels"].cuda()

            optimizer.zero_grad()
            cikti = model_temp(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            cikti.loss.backward()
            optimizer.step()
            scheduler.step()
            toplam += cikti.loss.item()
        son_loss = toplam / len(dl)

    bayesian_sonuclar.append({
        "lr": learning_rate,
        "epoch": epoch,
        "batch": batch_size,
        "loss": son_loss
    })

    print(f"  lr={learning_rate:.6f} epoch={epoch} batch={batch_size} → Loss={son_loss:.4f}")

    del model_temp
    torch.cuda.empty_cache()

    return -son_loss  # Bayesian maksimize eder, biz minimize istiyoruz

# Arama uzayı
pbounds = {
    "learning_rate": (5e-5, 3e-4),
    "epoch":         (3, 6),
    "batch_size":    (2, 8),
}

optimizer_bayes = BayesianOptimization(
    f=gpt2_egit,
    pbounds=pbounds,
    random_state=42,
    verbose=0
)

print("🔍 Bayesian Optimizasyon başlıyor (8 iterasyon)...\n")
optimizer_bayes.maximize(init_points=3, n_iter=5)

# Sonuçları göster
en_iyi_bayes = min(bayesian_sonuclar, key=lambda x: x["loss"])
print(f"\n{'='*50}")
print(f"BAYESIAN OPTİMİZASYON SONUCU")
print(f"{'='*50}")
print(f"  En iyi lr         : {en_iyi_bayes['lr']:.6f}")
print(f"  En iyi epoch      : {en_iyi_bayes['epoch']}")
print(f"  En iyi batch size : {en_iyi_bayes['batch']}")
print(f"  En iyi Loss       : {en_iyi_bayes['loss']:.4f}")

# Kaydet
with open("/content/drive/MyDrive/gpt2_bayesian_sonuclari.json", "w") as f:
    json.dump({
        "grid_search": gpt2_sonuclar,
        "bayesian": bayesian_sonuclar,
        "en_iyi": en_iyi_bayes
    }, f, indent=2, ensure_ascii=False)

print("\n✅ Tüm sonuçlar kaydedildi.")

# ── 3. KARŞILAŞTIRMA GRAFİĞİ ────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))

# Grid search sonuçları
isimler = [f"Grid-{s['deney']}" for s in gpt2_sonuclar]
losslar = [s["final_loss"] for s in gpt2_sonuclar]

# Bayesian sonuçları
for i, s in enumerate(bayesian_sonuclar):
    isimler.append(f"Bayes-{i+1}")
    losslar.append(s["loss"])

renkler_bar = ['#3498db']*3 + ['#e74c3c']*len(bayesian_sonuclar)
bars = ax.bar(isimler, losslar, color=renkler_bar, edgecolor='black', linewidth=0.5)

# En iyiyi vurgula
en_iyi_idx = np.argmin(losslar)
bars[en_iyi_idx].set_color('gold')
bars[en_iyi_idx].set_edgecolor('black')
bars[en_iyi_idx].set_linewidth(2)

ax.set_title("GPT-2 — Grid Search vs Bayesian Optimizasyon\nLoss Karşılaştırması",
             fontsize=12, fontweight='bold')
ax.set_ylabel("Final Training Loss")
ax.set_xlabel("Deney")
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

# Değerleri bar üstüne yaz
for bar, val in zip(bars, losslar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha='center', va='bottom', fontsize=8)

from matplotlib.patches import Patch
legend = [
    Patch(color='#3498db', label='Grid Search'),
    Patch(color='#e74c3c', label='Bayesian'),
    Patch(color='gold',    label='En iyi')
]
ax.legend(handles=legend)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/gpt2_optimizasyon_karsilastirma.png", dpi=150)
plt.show()

In [ ]:
!pip install nltk -q

import torch, math, json, numpy as np, nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import AutoTokenizer, AutoModelForCausalLM
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from google.colab import drive
drive.mount('/content/drive')

TEST_KELIMELERI = [
    "wifi", "suşi", "selfie", "bitcoin", "çiğköfte",
    "kargo", "şarj", "deadline", "metro", "market",
    "tiktok", "barista", "influencer", "netflix", "sipariş",
    "asansör", "navigasyon", "zihin", "iftira", "trip"
]

REFERANS_SIIRLER = [
    "sokakta yürürüm ben de insanlar gibi",
    "ne güzel şeydir yaşamak bu dünyada",
    "istanbul'u dinliyorum gözlerim kapalı",
    "garip benim garip istanbul garip hayat",
    "para yok cebimde ne yapayım"
]

def siir_uret_gpt2(model, tokenizer, kelime, max_deneme=3):
    prompt = f"### Talimat:\nSen Orhan Veli Kanık'sın. \"{kelime}\" kelimesini kullanarak şiir yaz.\n\n### Şiir:\n"
    inputs = tokenizer(prompt, return_tensors="pt")  # CPU

    for _ in range(max_deneme):
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.8,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        prompt_len = inputs["input_ids"].shape[1]
        siir = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
        dizeler = [d for d in siir.split('\n') if d.strip()]
        siir = '\n'.join(dizeler[:8])
        if kelime.lower() in siir.lower():
            return siir
    return siir

def kelime_kullanim_orani(siirler, kelimeler):
    return sum(1 for k, s in zip(kelimeler, siirler) if k.lower() in s.lower()) / len(kelimeler)

def perplexity_hesapla(model, tokenizer, metinler):
    model.eval()
    toplam, sayi = 0, 0
    for metin in metinler:
        enc = tokenizer(metin, return_tensors="pt",
                       truncation=True, max_length=512)  # CPU
        with torch.no_grad():
            cikti = model(**enc, labels=enc["input_ids"])
            toplam += cikti.loss.item()
            sayi += 1
    return math.exp(toplam / sayi)

def bleu_hesapla(siirler, referanslar):
    smoother = SmoothingFunction().method1
    skorlar = []
    for siir in siirler:
        hip = nltk.word_tokenize(siir.lower())
        refs = [nltk.word_tokenize(r.lower()) for r in referanslar]
        skorlar.append(sentence_bleu(refs, hip, smoothing_function=smoother))
    return np.mean(skorlar)

# CPU'da yükle
print("GPT-2 CPU'da yükleniyor...")
gpt2_tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
)
gpt2_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/gpt2_orhan_veli_modeli"
)  # CPU — .cuda() yok
print("✅ GPT-2 yüklendi.")

print("\n📊 GPT-2 metrikleri hesaplanıyor... (~40 dakika)\n")

gpt2_siirler = []
for i, kelime in enumerate(TEST_KELIMELERI, 1):
    siir = siir_uret_gpt2(gpt2_model, gpt2_tokenizer, kelime)
    gpt2_siirler.append(siir)
    durum = "✅" if kelime.lower() in siir.lower() else "❌"
    print(f"  [{i:02d}/20] {durum} {kelime}")

kko  = kelime_kullanim_orani(gpt2_siirler, TEST_KELIMELERI)
perp = perplexity_hesapla(gpt2_model, gpt2_tokenizer, gpt2_siirler)
bleu = bleu_hesapla(gpt2_siirler, REFERANS_SIIRLER)

gpt2_metrikler = {
    "model": "GPT-2 Türkçe",
    "kelime_kullanim_orani": round(kko, 4),
    "perplexity": round(perp, 2),
    "bleu": round(bleu, 4),
    "en_iyi_loss": 0.2153
}

print(f"\n{'='*45}")
print(f"  GPT-2 TÜRKÇE METRİKLERİ")
print(f"{'='*45}")
print(f"  Kelime Kullanım Oranı : %{kko*100:.1f}")
print(f"  Perplexity            : {perp:.2f}")
print(f"  BLEU Skoru            : {bleu:.4f}")
print(f"{'='*45}")

with open("/content/drive/MyDrive/gpt2_metrikler.json", "w") as f:
    json.dump(gpt2_metrikler, f, indent=2, ensure_ascii=False)

print("✅ GPT-2 metrikleri kaydedildi.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json

# Verileri yükle
with open("/content/drive/MyDrive/llama_metrikler.json") as f:
    llama = json.load(f)
with open("/content/drive/MyDrive/qwen_metrikler.json") as f:
    qwen = json.load(f)
with open("/content/drive/MyDrive/gpt2_metrikler.json") as f:
    gpt2 = json.load(f)

modeller = ["LLaMA-3 8B", "Qwen2.5 1.5B", "GPT-2 Türkçe"]
renkler  = ["#2ecc71", "#e74c3c", "#3498db"]

kko  = [llama["kelime_kullanim_orani"], qwen["kelime_kullanim_orani"], gpt2["kelime_kullanim_orani"]]
perp = [llama["perplexity"], qwen["perplexity"], gpt2["perplexity"]]
bleu = [llama["bleu"], qwen["bleu"], gpt2["bleu"]]
loss = [llama["en_iyi_loss"], qwen["en_iyi_loss"], gpt2["en_iyi_loss"]]

fig = plt.figure(figsize=(18, 14))
fig.suptitle("3 Model Karşılaştırması — Orhan Veli Şiir Üretimi",
             fontsize=16, fontweight='bold', y=0.98)

# ── 1. KKO Bar Chart ─────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
bars = ax1.bar(modeller, [k*100 for k in kko], color=renkler,
               edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, kko):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"%{val*100:.1f}", ha='center', fontsize=11, fontweight='bold')
ax1.set_title("Kelime Kullanım Oranı (%)\n(Yüksek = İyi)", fontweight='bold')
ax1.set_ylabel("%")
ax1.set_ylim(0, 115)
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=10)

# ── 2. Perplexity Bar Chart ───────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
bars2 = ax2.bar(modeller, perp, color=renkler,
                edgecolor='black', linewidth=0.7)
for bar, val in zip(bars2, perp):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{val:.2f}", ha='center', fontsize=11, fontweight='bold')
ax2.set_title("Perplexity\n(Düşük = İyi)", fontweight='bold')
ax2.set_ylabel("Perplexity")
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=10)

# ── 3. BLEU Bar Chart ─────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
bars3 = ax3.bar(modeller, bleu, color=renkler,
                edgecolor='black', linewidth=0.7)
for bar, val in zip(bars3, bleu):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
             f"{val:.4f}", ha='center', fontsize=11, fontweight='bold')
ax3.set_title("BLEU Skoru\n(Yüksek = İyi)", fontweight='bold')
ax3.set_ylabel("BLEU")
ax3.grid(axis='y', alpha=0.3)
ax3.tick_params(axis='x', rotation=10)

# ── 4. Radar Chart ────────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4, polar=True)

kategoriler = ['KKO', 'Perplexity\n(ters)', 'BLEU']
N = len(kategoriler)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

# Normalize et (0-1 arası)
max_perp = max(perp)
radar_verileri = {
    "LLaMA-3 8B":    [kko[0], 1 - perp[0]/max_perp, bleu[0]/max(bleu)],
    "Qwen2.5 1.5B":  [kko[1], 1 - perp[1]/max_perp, bleu[1]/max(bleu)],
    "GPT-2 Türkçe":  [kko[2], 1 - perp[2]/max_perp, bleu[2]/max(bleu)],
}

for (isim, deger), renk in zip(radar_verileri.items(), renkler):
    deger_kapatilmis = deger + deger[:1]
    ax4.plot(angles, deger_kapatilmis, 'o-', linewidth=2,
             color=renk, label=isim)
    ax4.fill(angles, deger_kapatilmis, alpha=0.1, color=renk)

ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(kategoriler, fontsize=10)
ax4.set_title("Radar Karşılaştırması\n(Normalize)", fontweight='bold', pad=20)
ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

# ── 5. Loss Karşılaştırması ───────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
bars5 = ax5.bar(modeller, loss, color=renkler,
                edgecolor='black', linewidth=0.7)
for bar, val in zip(bars5, loss):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.4f}", ha='center', fontsize=11, fontweight='bold')
ax5.set_title("En İyi Training Loss\n(Düşük = İyi — Modeller arası karşılaştırılamaz!)",
              fontweight='bold', fontsize=9)
ax5.set_ylabel("Loss")
ax5.grid(axis='y', alpha=0.3)
ax5.tick_params(axis='x', rotation=10)

# ── 6. Özet Tablo ─────────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')

tablo_veri = [
    ["LLaMA-3 8B",   "%90.0 🏆", "23.12",  "0.0076", "1.1181"],
    ["Qwen2.5 1.5B", "%50.0",    "21.17 🏆","0.0053", "1.8872"],
    ["GPT-2 Türkçe", "%85.0",    "28.69",  "0.0083 🏆","0.2153"],
]
sutunlar = ["Model", "KKO", "Perplexity", "BLEU", "Loss"]

tablo = ax6.table(
    cellText=tablo_veri,
    colLabels=sutunlar,
    cellLoc='center',
    loc='center',
    bbox=[0, 0.2, 1, 0.7]
)
tablo.auto_set_font_size(False)
tablo.set_fontsize(9)

for j in range(len(sutunlar)):
    tablo[0, j].set_facecolor('#2c3e50')
    tablo[0, j].set_text_props(color='white', fontweight='bold')

for i, renk in enumerate(["#d5f5e3", "#fadbd8", "#d6eaf8"]):
    for j in range(len(sutunlar)):
        tablo[i+1, j].set_facecolor(renk)

ax6.set_title("Özet Sonuç Tablosu", fontweight='bold')

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/model_karsilastirma_final.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Karşılaştırma grafiği kaydedildi.")